In [ ]:
from __future__ import annotations
import os
os.environ['HF_HOME'] = "~/.cache"

import argparse
from pathlib import Path

import cv2
import torch
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline
from peft import PeftModel
from PIL import Image


In [ ]:
args = {
    "structure_map": "",
    "controlnet_path": "",
    "lora_path": "",
    "output_path": "",
    "pretrained_model_name_or_path": "stabilityai/stable-diffusion-xl-base-1.0",
    "prompt": "macro photo of printed circuit board, green solder mask, copper traces, realistic industrial PCB",
    "negative_prompt": "",
    "num_inference_steps": 30,
    "guidance_scale": 6.0,
    "controlnet_conditioning_scale": 1.0,
    "lora_scale": 0.8,
    "height": 768,
    "width": 768,
    "seed": 42
}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weight_dtype = torch.float16 if device.type == "cuda" else torch.float32

controlnet = ControlNetModel.from_pretrained(
    args.controlnet_path,
    torch_dtype=weight_dtype,
)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    args.pretrained_model_name_or_path,
    controlnet=controlnet,
    torch_dtype=weight_dtype,
    low_cpu_mem_usage=True
)

pipe.unet = PeftModel.from_pretrained(pipe.unet, args.lora_path)
for module in pipe.unet.modules():
    if hasattr(module, "scaling"):
        for key in module.scaling:
            module.scaling[key] = args.lora_scale
pipe.unet.eval()

pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.enable_attention_slicing()


In [ ]:
structure = cv2.imread(args.structure_map, cv2.IMREAD_GRAYSCALE)
if structure is None:
    raise FileNotFoundError(f"Failed to read structure map: {args.structure_map}")
structure = cv2.resize(structure, (args.width, args.height))
structure_image = Image.fromarray(structure)

generator = torch.Generator(device=device.type).manual_seed(args.seed)
result = pipe(
    prompt=args.prompt,
    negative_prompt=args.negative_prompt or None,
    image=structure_image,
    num_inference_steps=args.num_inference_steps,
    guidance_scale=args.guidance_scale,
    controlnet_conditioning_scale=args.controlnet_conditioning_scale,
    height=args.height,
    width=args.width,
    generator=generator,
).images[0]

output_path = Path(args.output_path)
output_path.parent.mkdir(parents=True, exist_ok=True)
result.save(output_path)
print(f"Saved generated image to: {output_path}")